<a href="https://colab.research.google.com/github/soberbichler/Discourse-in-Spanish-flu-coverage_Notebook/blob/main/Gallica_API_SRU_IIIF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Accessing French Historical Newspapers from BnF using the Gallica API's

This notebook uses and adapts code provided by the Bibliothèque nationale de France (BNF), specifically by Jean-Philippe Moreux (Expert scientifique Gallica, jean-philippe.moreux@bnf.fr) and Arnaud Laborderie
(Chef de projet Gallica : exploitation des données pour la recherche, arnaud.laborderie@bnf.fr) as well as the project [MADOAP](https://modoap.huma-num.fr/)  (lead by Julien SCHUH , Université Paris Nanterre – CSLF EA 1586).

The GitHub repository of the project MADOAP containing the code that has been adapted for this notebook can be found [here](https://github.com/MODOAP?tab=repositories).

For documentation on how to use the SUR API see [here](https://https://api.bnf.fr/fr/api-gallica-de-recherche).

## This notebook was created for the following workflow:

1.	Getting identifiers using the SUR API
2.	Saving the identifiers in a DataFrame, together with the title of the documents, and date
3.	Downloading the txt files of the documents using the identifiers from the DataFrame
4.	Extracting the metadata from the txt files and creating a DataFrame that contains metadata and full text
5.	Limiting the full text to a certain context window for further processing
6.	Downloading the DataFrame as Excel file for further processing







# Using the SRU API to retrieve the identifiers for a search request at Gallica

Steps:


1.   Go to Gallica (https://gallica.bnf.fr/accueil/de/content/accueil-de?mode=desktop) and search for the documents that are relevant for you
2.   In the Gallica URL, copy the CQL query following the query= parameter
3.   Insert the CQL query below after "query = "

In this specific example I was searching for news on eartquakes in the newspaper: L'Humanité : journal socialiste quotidien.


In [1]:
# URL de base du service Gallica SRU
BASEURL = 'https://gallica.bnf.fr/SRU?version=1.2&operation=searchRetrieve&query='
# L'API SRU renvoie au maximum 50 documents par appel
MODULE = 50
# Mode d'agrégation des résultats : ne pas agréger les documents multivolumes
COLLAPSING = "false"

query = 'dc.title all "Humanité (L)" and (( text adj "tremblement de terre" or text adj "tremblements de terre" or text adj "séisme" or text adj "sismiques" or text adj "secousses" or text adj "messina" or text adj "calabre")) and (dc.type all "fascicule") and (gallicapublication_date>="1908/01/01" and gallicapublication_date<="1909/06/30") sortby dc.title/sort.ascending'
#query = '(text adj "Messine" or text adj "messine" ) and (dc.type all "fascicule") and (gallicapublication_date>="1908/12/01" and gallicapublication_date<="1909/06/31")&suggest=10&keywords=Messine messine'
#query='(((((text adj "tremblement de terre" or text adj "Tremblement de terre" ) or text adj "tremblements de terre" ) or text adj "Tremblements de terre" ) or text adj "séisme" ) or text adj "Séisme" ) or text adj " ) or text adj "Sismiques" ) or text adj "sismiques" ) or text adj "Secousses" ) or text adj "secousses") and (dc.type all "fascicule") and (gallicapublication_date>="1908/12/28" and gallicapublication_date<="1909/12/31")&suggest=10&keywords=tremblement de terre Tremblement de terre tremblements de terre Tremblements de terre séisme Séisme'
# construire la reuqête avec ses paramètres
req_url = "".join([BASEURL, query, "&maximumRecords=", str(MODULE), "&collapsing=", COLLAPSING])


import requests

# Appel de l'API SRU avec le système de pagination : i = [1-n]
def getXML_from_SRU(indice):
    req = "".join([req_url,"&startRecord=",str(indice)])
    print ("... appel de l'API Gallica SRU : ", req)
    temp = requests.get(req)
    #print (temp.status_code)
    if temp.status_code != 200:
        print ("... oups, erreur API:", temp.status_code)
        return
    else:
        return temp

print ("... extraction de la page de résultats #", 1)
r = getXML_from_SRU(1)
if r is not None:
    print (r.text[0:1000])


... extraction de la page de résultats # 1
... appel de l'API Gallica SRU :  https://gallica.bnf.fr/SRU?version=1.2&operation=searchRetrieve&query=dc.title all "Humanité (L)" and (( text adj "tremblement de terre" or text adj "tremblements de terre" or text adj "séisme" or text adj "sismiques" or text adj "secousses" or text adj "messina" or text adj "calabre")) and (dc.type all "fascicule") and (gallicapublication_date>="1908/01/01" and gallicapublication_date<="1909/06/30") sortby dc.title/sort.ascending&maximumRecords=50&collapsing=false&startRecord=1
<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<srw:searchRetrieveResponse xmlns:ns6="http://gallica.bnf.fr/namespaces/gallica/" xmlns:diag="http://www.loc.gov/zing/srw/diagnostic/" xmlns:oai_dc="http://www.openarchives.org/OAI/2.0/oai_dc/" xmlns:srw="http://www.loc.gov/zing/srw/" xmlns:dc="http://purl.org/dc/elements/1.1/">
    <srw:version>1.2</srw:version>
    <srw:echoedSearchRetrieveRequest>
        <srw:query>dc.title al

In [2]:
# @markdown #### Check the number of resuts
from bs4 import BeautifulSoup as bs
bs_content = bs(r.text, "xml") # bizarre : avec le parser xml, on perd la fin du contenu XML ...
#bs_content = bs(r.text, "lxml") # par contre lxml ne fonctionne pas sur srw:numberOfRecords !

# Extraire le nombre de résultats
n_docs = int(bs_content.find("srw:numberOfRecords").getText())
print("Nombre de résultats : ", n_docs)

Nombre de résultats :  99


In [3]:
# @markdown #### Get a DataFrame with title, identifier, and date of the documents
from bs4 import BeautifulSoup
import pandas as pd
bs_content = bs(r.text, "lxml")

# si besoin, on pagine la suite
if n_docs > MODULE:
    print (n_docs)
    # Ajouter au document xml toutes les réponses des autres pages
    for i in range(1, int(n_docs/MODULE)+1):
        print ("Extraction de la page de résultats #", i+1)
        print ("    startRecord : ", i*MODULE+1)
        temp = getXML_from_SRU(i*MODULE+1)
        temp_xml = bs(temp.text, "lxml")
        bs_content.append(temp_xml)

print ("------------")
print("Nbre total d'identifiants de document : ",len(bs_content.find_all("dc:identifier")))


#for ark in bs_content.find_all("dc:title"):
 #   titles = ark.get_text()


titles = [ark.get_text() for ark in bs_content.find_all("dc:title")]
id = [ark.get_text() for ark in bs_content.find_all("dc:identifier")]
date = [ark.get_text() for ark in bs_content.find_all("dc:date")]
#language = [ark.get_text() for ark in bs_content.find_all("dc:language")]

# Create a DataFrame from the extracted titles
df = pd.DataFrame({'Title': titles,'ID': id, 'Date': date})
df['ID'] = df['ID'].str.split('12148/').str[1]

#df_gazette= df[df['Title'].str.contains('Gazette', case=False)]  # Case-insensitive search



# Display the DataFrame

df

/tmp/ipython-input-3199782366.py:4: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  bs_content = bs(r.text, "lxml")


99
Extraction de la page de résultats # 2
    startRecord :  51
... appel de l'API Gallica SRU :  https://gallica.bnf.fr/SRU?version=1.2&operation=searchRetrieve&query=dc.title all "Humanité (L)" and (( text adj "tremblement de terre" or text adj "tremblements de terre" or text adj "séisme" or text adj "sismiques" or text adj "secousses" or text adj "messina" or text adj "calabre")) and (dc.type all "fascicule") and (gallicapublication_date>="1908/01/01" and gallicapublication_date<="1909/06/30") sortby dc.title/sort.ascending&maximumRecords=50&collapsing=false&startRecord=51


/tmp/ipython-input-3199782366.py:14: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  temp_xml = bs(temp.text, "lxml")


------------
Nbre total d'identifiants de document :  99


,Title,ID,Date
0,L'Humanité : journal socialiste quotidien,bpt6k251617q,1908-03-28
1,L'Humanité : journal socialiste quotidien,bpt6k2519613,1909-03-15
2,L'Humanité : journal socialiste quotidien,bpt6k251656q,1908-05-06
3,L'Humanité : journal socialiste quotidien,bpt6k2516136,1908-03-24
4,L'Humanité : journal socialiste quotidien,bpt6k2519558,1909-03-09
...,...,...,...
94,L'Humanité : journal socialiste quotidien,bpt6k252055r,1909-06-18
95,L'Humanité : journal socialiste quotidien,bpt6k251894z,1909-01-07
96,L'Humanité : journal socialiste quotidien,bpt6k251895b,1909-01-08
97,L'Humanité : journal socialiste quotidien,bpt6k251877d,1908-12-21


# Extract the documents using their identifiers and create a DataFrame containing the full text and metadata of the extracted documents

First, get the text files of the documents and store them in your content file:


In [4]:
arks = df['ID'].tolist()
print(arks)

['bpt6k251617q', 'bpt6k2519613', 'bpt6k251656q', 'bpt6k2516136', 'bpt6k2519558', 'bpt6k251911c', 'bpt6k252049x', 'bpt6k2516997', 'bpt6k251942s', 'bpt6k251906x', 'bpt6k251615z', 'bpt6k251893k', 'bpt6k251889h', 'bpt6k2518973', 'bpt6k252001h', 'bpt6k251839s', 'bpt6k252050v', 'bpt6k2519469', 'bpt6k251690t', 'bpt6k2519961', 'bpt6k2516827', 'bpt6k252054c', 'bpt6k251705x', 'bpt6k251959s', 'bpt6k251933t', 'bpt6k2516573', 'bpt6k251800b', 'bpt6k2519079', 'bpt6k252032x', 'bpt6k252025q', 'bpt6k2517684', 'bpt6k2520339', 'bpt6k252048j', 'bpt6k2516615', 'bpt6k2518583', 'bpt6k2519257', 'bpt6k251926m', 'bpt6k252067v', 'bpt6k251908p', 'bpt6k2518884', 'bpt6k251989t', 'bpt6k2515839', 'bpt6k2518672', 'bpt6k251902d', 'bpt6k251685c', 'bpt6k2518926', 'bpt6k251912r', 'bpt6k251935k', 'bpt6k251857q', 'bpt6k251749t', 'bpt6k251891t', 'bpt6k2517603', 'bpt6k251658g', 'bpt6k2520004', 'bpt6k2518850', 'bpt6k251848r', 'bpt6k252062z', 'bpt6k2519092', 'bpt6k252036f', 'bpt6k251896q', 'bpt6k2519401', 'bpt6k251917n', 'bpt6k2

In [ ]:
journals_to_extract = ["La Dépêche : journal quotidien"]

df = df[df['Title'].isin(journals_to_extract)]


df = df.drop_duplicates(subset="ID", keep="first")

df

In [ ]:

import pandas as pd
from time import sleep, time
!pip install xmltodict
import xmltodict
import shutil
import requests
from bs4 import BeautifulSoup
from openpyxl import load_workbook
import urllib.request, urllib.error, urllib.parse
from urllib.error import HTTPError, URLError
from xml.etree import ElementTree as ET
from google.colab import drive
import os
import json
from tqdm import tqdm
import re
import unicodedata
import sqlite3

# Préparation et synchronisation d'un Google Drive
if not os.path.exists("/content/drive/MyDrive/"):
    drive.mount('/content/drive/')

erreurs = []
sans_ressource = []

# IIIF v3
region = "full"
max_size = "max"
rotation = "0"
quality = "default"
format = "jpg"

# API #
IIIF_pres_BASEURL = 'https://openapi.bnf.fr/iiif/presentation/v3/ark:/12148/'
IIIF_img_BASEURL = 'https://openapi.bnf.fr/iiif/image/v3/ark:/12148/'
OAI_BASEURL = "https://gallica.bnf.fr/services/OAIRecord?ark="
CAT_BASEURL = "https://catalogue.bnf.fr/ark:/12148/"
PAGINATION_BASEURL = 'https://gallica.bnf.fr/services/Pagination?ark='
GALLICA_BASEURL = 'https://gallica.bnf.fr/ark:/12148/'

def nb_pages(manifeste):
    with open(manifeste) as f:
        dico = json.load(f)
    nb_pages = len(dico["sequences"][0]["canvases"])
    return nb_pages

def remove_accents(s):
    return ''.join((c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn'))

def normalisation_titre(titre):
    titre = remove_accents(titre)
    titre = re.sub('[^a-zA-Z0-9- ]', '', titre)
    titre = re.sub('[ ]', '_', titre)
    return "_".join(titre.split("_")[:6])

def get_title(ark):
    try:
        s = requests.get(OAI_BASEURL + ark, stream=True)
        bibliodico = xmltodict.parse(s.text)
        titre = bibliodico["results"]["title"]
        titre = normalisation_titre(titre)
    except:
        print("# Impossible d'extraire le titre de l'OAI ! #")
        titre = ark
    return titre

def page_courante(ark, page, pagination):
    ordre = pagination["livre"]["pages"]["page"][int(page-1)]["ordre"]
    numero = pagination["livre"]["pages"]["page"][int(page-1)]["numero"]
    return ordre, numero

def bnf2gall(arkbnf):
    url = CAT_BASEURL + str(arkbnf)
    s = requests.get(url, stream=True)
    html = BeautifulSoup(s.content, "lxml-xml")
    for link in html.findAll('a', {'class': 'exemplaire-action-visualiser'}):
        ark = link['href'].split("/")[-1]
        return ark

# Obtention des informations de pagination
def paginationDL(ark):
    try:
        url = "".join([PAGINATION_BASEURL, ark])
        s = requests.get(url, stream=True)
        pagination = str(BeautifulSoup(s.content, "lxml-xml"))
        paginationdic = xmltodict.parse(pagination)
        nb_pages = int(paginationdic["livre"]["structure"]["nbVueImages"])
        ocr = paginationdic["livre"]["structure"]["hasContent"]
        toc = paginationdic["livre"]["structure"]["hasToc"]
        return paginationdic, nb_pages, ocr, toc
    except:
        print(url)
        print("# La pagination n'a pas été téléchargée ! #")
        return (None, 0, 0, 0)

###########################################
# Définition des fonctions de téléchargement
def altoDL(ark, page):
    url = "".join([IIIF_pres_BASEURL, ark, '/f', str(page), '/alto.xml'])
    if page == 1:
        print(url)
    nom_fichier = ark + "_" + str(page) + ".xml"
    try:
        urllib.request.urlretrieve(url, nom_fichier)
        return nom_fichier
    except (HTTPError, URLError) as erreur:
        erreurs.append(ark + ";" + str(page) + ";" + url + ";" + str(erreur.reason))
        print(" code erreur : ", erreur)
        return erreur.reason

def ocr_jsonDL(ark, page):
    url = "".join([IIIF_pres_BASEURL, ark, '/f', str(page), '/annotationpage/supplementing.json'])
    if page == 1:
        print(url)
    nom_fichier = ark + "_" + str(page) + ".json"
    try:
        urllib.request.urlretrieve(url, nom_fichier)
        return nom_fichier
    except (HTTPError, URLError) as erreur:
        erreurs.append(ark + ";" + str(page) + ";" + url + ";" + str(erreur.reason))
        print(" code erreur : ", erreur)
        return erreur.reason

def txtDL(ark, nb_pages):
    url = GALLICA_BASEURL + str(ark) + "/f1n" + str(nb_pages) + ".texteBrut"
    nom_fichier = ark + ".txt"
    print(url)
    try:
        page = urllib.request.urlopen(url).read().decode('utf-8')
    except (HTTPError, URLError) as erreur:
        erreurs.append(ark + ";;" + url + ";" + str(erreur.reason))
        print(" code erreur : ", erreur)
        return erreur.reason
    page = str(page)
    soup = BeautifulSoup(page, "html.parser")
    p_tags = soup.find_all("p")
    with open(nom_fichier, 'w') as f:
        for tag in p_tags:
            f.write(str(tag.text) + '\n')
    return nom_fichier

def pdfDL(ark):
    try:
        url = GALLICA_BASEURL + str(ark) + ".pdf"
        nom_fichier = ark + ".pdf"
        print(url)
        urllib.request.urlretrieve(url, nom_fichier)
    except (HTTPError, URLError) as erreur:
        erreurs.append(ark + ";;" + url + ";" + str(erreur.reason))
        print(" code erreur : ", erreur)
        return erreur.reason

def manifesteDL(ark):
    try:
        url = IIIF_pres_BASEURL + str(ark) + "/manifest.json"
        nom_fichier = ark + "-manifest.json"
        print(url)
        urllib.request.urlretrieve(url, nom_fichier)
        return nom_fichier
    except (HTTPError, URLError) as erreur:
        erreurs.append(ark + ";;" + url + str(erreur.reason))
        print(" code erreur : ", erreur)
        return erreur.reason

def pageDL(ark, page, taille):
    url = "".join([IIIF_img_BASEURL, ark, '/f', str(page), '/', region, '/', taille, '/', rotation, '/', quality, '.', format])
    nom_fichier = ark + "_" + str(page) + ".jpg"
    if page == 1:
        print(url)
    try:
        urllib.request.urlretrieve(url, nom_fichier)
        return nom_fichier
    except (HTTPError, URLError) as erreur:
        erreurs.append(ark + ";" + str(page) + ";" + url + str(erreur.reason))
        print(" code erreur : ", erreur)
        return erreur.reason

def tocDL(ark):
    try:
        url = IIIF_pres_BASEURL + str(ark) + "/structure/toc.json"
        nom_fichier = ark + "-toc.json"
        print(nom_fichier)
        print(url)
        urllib.request.urlretrieve(url, nom_fichier)
        return nom_fichier
    except (HTTPError, URLError) as erreur:
        erreurs.append(ark + ";;" + url + str(erreur.reason))
        print(" code erreur : ", erreur)
        return erreur.reason


chemin_destination = "/content/drive/MyDrive/Colab Notebooks/OUT"
format_telechargement = "txt"
taille_images = 20

if taille_images == 100:
    taille_images = max_size
else:
    taille_images = "pct:" + str(taille_images)

try:
    if not os.path.exists(chemin_destination):
        os.makedirs(chemin_destination)
except:
    print("# Le chemin de destination est incorrect #")
    quit()

request_count = 0
start_time = time()

from time import sleep, time

arks = df['ID'].tolist()
print("Format demandé : ", format_telechargement)
print("----------------------")

# Initialize timing and request count variables outside the loop
start_time = time()
request_count = 0

start_time = time()
request_count = 0


for ark in arks:
    # Your request code here
    # Simulating a request
    request_count += 1

    if request_count >= 5:
        sleep(61)
        start_time = time()
        request_count = 0


    print("\nIdentifiant du document : ", ark)
    titre_tmp = get_title(ark)
    if not os.path.exists(titre_tmp):
        os.makedirs(titre_tmp)
        print("... création du dossier : ", titre_tmp)
    os.chdir(titre_tmp)
    print("... enregistré dans : ", titre_tmp)

    pagination, nb_pages, ocr, toc = paginationDL(ark)
    if nb_pages == 0:
        erreurs.append(ark + ";;;" + "no_pagination")
        continue

    if format_telechargement == "jpg":
        print(" nombre de pages : ", str(nb_pages))
        print(" taille d'image : ", taille_images)
        for page in tqdm(range(nb_pages)):
            page += 1
            reponse_doc = pageDL(ark, page, taille_images)

    elif format_telechargement == "pdf":
        pdfDL(ark)

    elif format_telechargement == "txt":
        if ocr == "true":
            reponse_doc = txtDL(ark, nb_pages)
        else:
            print(" -> sans OCR ")
            sans_ressource.append(ark + ";;;" + "no_ocr")

    elif format_telechargement == "json":
        if ocr == "true":
            print(" nombre de pages : ", str(nb_pages))
            for page in tqdm(range(nb_pages)):
                page += 1
                reponse_doc = ocr_jsonDL(ark, page)
        else:
            print(" -> sans OCR ")
            sans_ressource.append(ark + ";;;" + "no_ocr")

    elif format_telechargement == "alto":
        if ocr == "true":
            print(" nombre de pages : ", str(nb_pages))
            for page in tqdm(range(nb_pages)):
                page += 1
                reponse_doc = altoDL(ark, page)
        else:
            print(" -> sans OCR ")
            sans_ressource.append(ark + ";;;" + "no_ocr")

    elif format_telechargement == "toc":
        if toc == "true":
            reponse_doc = tocDL(ark)
        else:
            print(" -> sans TdM")
            sans_ressource.append(ark + ";;;" + "no_toc")

    elif format_telechargement == "manifest":
        manifesteDL(ark)

    os.chdir("../")
    request_count += 1

erreurs = set(erreurs)
with open("erreurs.txt", "w") as err:
    err.write("=============================\n")
    err.write("Erreurs : \n")
    err.write("=============================\n")
    for ark in erreurs:
        err.write(ark)
        err.write("\n")
    err.write("=============================\n")
    err.write("Documents sans ressource : \n")
    err.write("=============================\n")
    for ocrless in sans_ressource:
        err.write(ocrless)
        err.write("\n")

print("Format demandé : ", format_telechargement)
print("----------------------")
print("Documents non téléchargés : {0}".format(len(erreurs)))
print("Documents sans la ressource demandée : {0}".format(len(sans_ressource)))
print("(ces documents sont listés dans le fichier erreurs.txt)")


Mounted at /content/drive/
Format demandé :  txt
----------------------

Identifiant du document :  bpt6k251617q
# Impossible d'extraire le titre de l'OAI ! #
... création du dossier :  bpt6k251617q
... enregistré dans :  bpt6k251617q
https://gallica.bnf.fr/ark:/12148/bpt6k251617q/f1n4.texteBrut

Identifiant du document :  bpt6k2519613
... création du dossier :  LHumanite__journal_socialiste_quotidien
... enregistré dans :  LHumanite__journal_socialiste_quotidien
https://gallica.bnf.fr/ark:/12148/bpt6k2519613/f1n4.texteBrut

Identifiant du document :  bpt6k251656q
... enregistré dans :  LHumanite__journal_socialiste_quotidien
https://gallica.bnf.fr/ark:/12148/bpt6k251656q/f1n4.texteBrut

Identifiant du document :  bpt6k2516136
... enregistré dans :  LHumanite__journal_socialiste_quotidien
https://gallica.bnf.fr/ark:/12148/bpt6k2516136/f1n4.texteBrut

Identifiant du document :  bpt6k2519558
... enregistré dans :  LHumanite__journal_socialiste_quotidien
https://gallica.bnf.fr/ark:/12148/

In [ ]:
import os
import pandas as pd

def parse_text(text):
    data = {}
    lines = text.split('\n')
    text_content = []
    capture_text = False

    for line in lines:
        if "Le texte affiché peut comporte" in line:
            capture_text = True
            text_content.append(line.split("Le texte affiché peut comporter un certain nombre d'erreurs", 1)[1].strip())
        elif capture_text:
            text_content.append(line)
        else:
            if "Titre :" in line:
                data['Titre'] = line.split(" : ")[1].strip()
            elif "Auteur :" in line:
                data['Auteur'] = line.split(" : ")[1].strip()
            elif "Éditeur :" in line:
                data['Éditeur'] = line.split(" : ")[1].strip()
            elif "Date d'édition :" in line:
                data['Date d\'édition'] = line.split(" : ")[1].strip()
            elif "Contributeur :" in line:
                contributors = data.get('Contributeurs', [])
                contributors.append(line.split(" : ")[1].strip())
                data['Contributeurs'] = contributors
            elif "Notice du catalogue :" in line:
                notices = data.get('Notice du catalogue', [])
                notices.append(line.split(" : ")[1].strip())
                data['Notice du catalogue'] = notices
            elif "Type :" in line:
                types = data.get('Type', [])
                types.append(line.split(" : ")[1].strip())
                data['Type'] = types
            elif "Langue :" in line:
                data['Langue'] = line.split(" : ")[1].strip()
            elif "Format :" in line:
                data['Format'] = line.split(" : ")[1].strip()
            elif "Description :" in line:
                descriptions = data.get('Description', [])
                descriptions.append(line.split(" : ")[1].strip())
                data['Description'] = descriptions
            elif "Droits :" in line:
                droits = data.get('Droits', [])
                droits.append(line.split(" : ")[1].strip())
                data['Droits'] = droits
            elif "Identifiant :" in line:
                data['Identifiant'] = line.split(" : ")[1].strip()
            elif "Source :" in line:
                data['Source'] = line.split(" : ")[1].strip()
            elif "Conservation numérique :" in line:
                data['Conservation numérique'] = line.split(" : ")[1].strip()
            elif "Date de mise en ligne :" in line:
                data['Date de mise en ligne'] = line.split(" : ")[1].strip()

    if text_content:
        data['text'] = '\n'.join(text_content).strip()

    return data

# Specify the base directory containing the folders
base_directory = '/content/LHumanite__journal_socialiste_quotidien'  # Update this path to your actual directory

# Initialize an empty list to hold the parsed data from all files
all_data = []

# Traverse through each folder and subfolder
for root, dirs, files in os.walk(base_directory):
    # Filter for .txt files
    txt_files = [file for file in files if file.endswith('.txt')]

    # Process each .txt file
    for txt_file in txt_files:
        file_path = os.path.join(root, txt_file)

        # Read and parse each file
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            parsed_data = parse_text(content)
            all_data.append(parsed_data)

# Convert the list of dictionaries into a DataFrame
df = pd.DataFrame(all_data)

# Display the DataFramepr
print(len(df))
df.to_csv('humanite.csv')
df


In [ ]:
import pandas as pd

# Specify the path to your CSV file
csv_file_path = '/content/le_petit_parisien.csv'  # Replace with the actual path to your CSV file

df = pd.read_csv(csv_file_path)
df

### Split into chunks

In [ ]:
import pandas as pd
import numpy as np

# Function to split text into chunks
def split_text_into_chunks(text, num_chunks=6):
    # Calculate the approximate chunk size
    words = text.split()  # Split text into words
    chunk_size = len(words) // num_chunks  # Approximate size for each chunk

    # Split into chunks and re-join into strings
    chunks = [' '.join(words[i*chunk_size:(i+1)*chunk_size]) for i in range(num_chunks)]

    # If there are remaining words, add them to the last chunk
    if len(words) % num_chunks != 0:
        chunks[-1] += ' ' + ' '.join(words[num_chunks * chunk_size:])

    return chunks

# Expand DataFrame by creating new rows for each chunk
expanded_rows = []
for _, row in df.iterrows():
    chunks = split_text_into_chunks(row['text'])
    for i, chunk in enumerate(chunks):
        new_row = row.copy()  # Copy the original row data
        new_row['text'] = chunk  # Replace text with the chunk
        new_row['Chunk'] = i + 1  # Add Chunk label (1 to 6)
        expanded_rows.append(new_row)

# Create new DataFrame from expanded rows
df_expanded = pd.DataFrame(expanded_rows)

# Display the expanded DataFrame
print(df_expanded)


## Creating Context-Window

In [ ]:
def extract_context(keywords, text, tokens_before, tokens_after):
    # transform text to lower letters
    lower_text = text.lower()

    # find all keyword positions (case-insensitive)
    keyword_positions = []
    for keyword in keywords:
        keyword_start = lower_text.find(keyword)
        while keyword_start != -1:
            keyword_positions.append(keyword_start)
            keyword_start = lower_text.find(keyword, keyword_start + len(keyword))

    if not keyword_positions:
        return "Keywords not found in text."

    # Determine start and end of context window
    first_occurrence = min(keyword_positions)
    last_occurrence = max(keyword_positions)

    start_index = max(0, first_occurrence - tokens_before)
    end_index = min(len(text), last_occurrence + tokens_after)

    # Extract context from original text
    context = text[start_index:end_index]

    return context

# keywords
keywords = ["tremblement", "tremblements", "messine", "séisme", "calabre", "sismiques", "secousses"]

# append context column
df['context'] = [extract_context(keywords, row['text'], 2000, 3000) for _, row in df.iterrows()]
df['context_small'] = [extract_context(keywords, row['text'], 1000, 2000) for _, row in df.iterrows()]

df


In [ ]:
# Filter out rows where 'context' column has the specific message
df = df[df['context'] != "Keywords not found in text."].reset_index(drop=True)

print(len(df))
df.head()

In [ ]:
# save the dataframe as an excel file
df.to_csv('Humanite_Text.csv')